# Feature Engineering

In [1]:
# ============================================================
# Load latest compiled macro-regime data snapshot
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

EXPORT_BASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "macro_regime_v2"
)

LATEST_PATH = EXPORT_BASE / "latest.json"

if not LATEST_PATH.exists():
    raise FileNotFoundError(
        "No compiled data snapshot found. "
        "Run Data_Engineering.ipynb first."
    )

latest_pointer = json.loads(
    LATEST_PATH.read_text(encoding="utf-8")
)

SNAPSHOT_DIR = (
    EXPORT_BASE
    / latest_pointer["snapshot_id"]
)

MANIFEST_PATH = SNAPSHOT_DIR / "manifest.json"

manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)


def file_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def load_artifact(
    name: str,
    verify_hash: bool = True,
) -> pd.DataFrame:
    if name not in manifest["artifacts"]:
        raise KeyError(
            f"Artifact not found in manifest: {name}"
        )

    metadata = manifest["artifacts"][name]
    file_path = SNAPSHOT_DIR / metadata["file"]

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    if verify_hash:
        actual_hash = file_sha256(file_path)

        if actual_hash != metadata["sha256"]:
            raise ValueError(
                f"Checksum failure for {name}"
            )

    frame = pd.read_parquet(file_path)

    expected_shape = (
        metadata["rows"],
        metadata["columns"],
    )

    if frame.shape != expected_shape:
        raise ValueError(
            f"{name}: expected {expected_shape}, "
            f"received {frame.shape}"
        )

    return frame


compiled_macro_events = load_artifact(
    "compiled_macro_events"
)

weekly_macro_raw = load_artifact(
    "weekly_macro_raw"
)

weekly_macro_compiled = load_artifact(
    "weekly_macro_compiled"
)

weekly_macro_observation_dates = load_artifact(
    "weekly_macro_observation_dates"
)

weekly_macro_available_dates = load_artifact(
    "weekly_macro_available_dates"
)

weekly_macro_observation_age_days = load_artifact(
    "weekly_macro_observation_age_days"
)

weekly_macro_publication_age_days = load_artifact(
    "weekly_macro_publication_age_days"
)

weekly_macro_stale = load_artifact(
    "weekly_macro_stale"
).astype(bool)

weekly_market_compiled = load_artifact(
    "weekly_market_compiled"
)

weekly_target_prices = load_artifact(
    "weekly_target_prices"
)

decision_calendar = load_artifact(
    "decision_calendar"
)

dataset_contracts = load_artifact(
    "dataset_contracts"
)

macro_staleness_limits = load_artifact(
    "macro_staleness_limits"
).iloc[:, 0]

compiled_coverage = load_artifact(
    "compiled_coverage"
)

sample_audit = load_artifact(
    "sample_audit"
)


compiled_samples = {}

for sample_name, sample_metadata in manifest[
    "samples"
].items():
    compiled_samples[sample_name] = {
        "name": sample_name,
        "start": pd.Timestamp(sample_metadata["start"]),
        "end": pd.Timestamp(sample_metadata["end"]),
        "inputs": load_artifact(
            f"sample_{sample_name}_inputs"
        ),
        "targets": load_artifact(
            f"sample_{sample_name}_targets"
        ),
        "calendar": load_artifact(
            f"sample_{sample_name}_calendar"
        ),
        "valid_week": load_artifact(
            f"sample_{sample_name}_valid_week"
        )["valid_week"].astype(bool),
        "required_series": sample_metadata[
            "required_series"
        ],
    }


core_panel = compiled_samples["core"]["inputs"]
core_target_prices = compiled_samples["core"]["targets"]
core_calendar = compiled_samples["core"]["calendar"]
core_valid_week = compiled_samples["core"]["valid_week"]

assert core_panel.index.equals(core_target_prices.index)
assert core_panel.index.equals(core_valid_week.index)

print(
    f"Loaded snapshot {manifest['snapshot_id']} | "
    f"core panel {core_panel.shape[0]:,} × "
    f"{core_panel.shape[1]:,}"
)

display(sample_audit)
display(core_panel.tail())

Loaded snapshot as_of_2026-09-10 | core panel 656 × 18


,start,end,calendar_weeks,input_series,required_series,valid_weeks,valid_week_pct,longest_invalid_run_weeks,target_complete_weeks
sample,,,,,,,,,
core,2014-02-14,2026-09-04,656,18,8,649,98.932927,6,656
funding_2018,2018-04-06,2026-09-04,440,22,12,434,98.636364,6,440
realtime_2019,2019-09-06,2026-09-04,366,20,10,360,98.360656,6,366
reserves_2021,2021-07-30,2026-09-04,267,19,9,261,97.752809,6,267
credit_2023,2023-09-15,2026-09-04,156,19,9,150,96.153846,6,156


,INDPRO,ICSA,unrate,cleveland_cpi_nowcast_mom,cleveland_core_cpi_nowcast_mom,cpi_index_sa,T10YIE,infl_5y5y,NFCI,ANFCI,fedfunds,DFII10,massive_treasury_yield_3_month,massive_treasury_yield_2_year,massive_treasury_yield_10_year,EPU,oil_wti,VIX
decision_date,,,,,,,,,,,,,,,,,,
2026-08-07,102.6395,199000.0,4.1,0.377373,0.203008,332.568,2.25,2.28,-0.529,-0.543,3.63,2.43,3.87,4.19,4.65,173.93,81.96,14.90
2026-08-14,102.6395,209000.0,4.1,0.345617,0.203342,332.813,2.27,2.30,-0.549,-0.579,3.63,2.39,3.86,4.17,4.68,219.75,84.77,14.25
2026-08-21,102.9939,206000.0,4.1,0.347334,0.203342,332.813,2.34,2.34,-0.559,-0.587,3.63,2.35,3.88,4.24,4.74,301.86,86.48,15.13
2026-08-28,102.9939,203000.0,4.1,0.355918,0.203342,332.813,2.31,2.32,-0.566,-0.576,3.63,2.34,3.90,4.34,4.73,131.19,83.90,14.43
2026-09-04,102.9939,206000.0,4.1,0.382154,0.194477,332.813,2.35,2.33,-0.558,-0.582,3.63,2.42,3.91,4.37,4.78,205.69,91.48,14.53
